In [1]:
print('Importing libraries...')
import json
import os
from pathlib import Path
import warnings 

import torch
from torch.utils.data import DataLoader
import torch.optim as optim

from scripts.env import set_deterministic_behaviour
from scripts.dataset import CachedDataset
from scripts.refactoring_caching import cache_validation
from scripts.metrics import update_model_output_dict, calculate_metrics
from scripts.model import build_model
from scripts.helper_functions import get_parameter_groups
from scripts.bbkl_loss import total_bb_loss
warnings.filterwarnings("ignore")

PWD = Path.cwd()
print(f"PWD: {PWD}")

Importing libraries...
PWD: /home/franek/Documents/python_code/BetaBinomialLoss


/home/franek/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# General
SEED = 0
EXPERIMENT_NAME = f"evid_bbloss_wkl_wprior_gated_swin_{SEED}"

# Data Paths
ANNOTATIONS_PATH = PWD / 'config/reformatted_annotations_frames.json'
DATASET_PATH = PWD.parent / 'dataset/endoscapes/'
CACHED_IMAGES_PATH = PWD / 'cached_images_temporal'

# Parameters to create the dataset
FORCE_RECACHE = False
TEMPORAL = True

# Model variables
BACKBONE_DICT = {'dino':    None,   #{'kwargs':          {'lora':                    None,
                                    #                     'partial_training_blocks': 6,
                                    #                     'imnt_weights_path':       './weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'},
                                    # 'backbone_weights': 'weights/DinoV3_encoder_pretraining_0.pt'},
                 'swin':    {'kwargs':              {'frozen_stages':   2},
                             'backbone_weights':    'weights/SwinV2_encoder_pretraining_0.pt'}}

CLASSIFIER_DICT = {'temporal_processing':   'gated_pooling',
                   'head':                  'evidential'}

# Training specific
BATCH_SIZE = 8

TEMPORAL_PROC_LR = 1e-4
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-2

BB_LOSS_VARIABLES = {'weights_bb_loss': {   'C1': (3.1985, 0.5926),     # 'C1': (1, 1),
                                            'C2': (4.4615, 0.5631),     # 'C2': (1, 1),
                                            'C3': (2.7952, 0.6089)},    # 'C3': (1, 1)
                     'use_kl': True,
                     'prior_alpha': {'nu': 2,
                                     'pi_C1': 0.1563,
                                     'pi_C2': 0.1121,   
                                     'pi_C3': 0.1789}}

EPOCHS = 10

In [3]:
##############################################################################################
##############################################################################################
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    device = 'cpu'

set_deterministic_behaviour(SEED)

# These are declared in scripts/refactoring_caching.py
# IMAGE_SIZE = (384, 384)
# DATASET_MEAN = (0.454315, 0.290313, 0.299898)
# DATASET_STD = (0.167318, 0.156652, 0.150197)

cache_validation(   DATASET_PATH,
                    CACHED_IMAGES_PATH,
                    ANNOTATIONS_PATH,
                    temporal = TEMPORAL,
                    force_recache = FORCE_RECACHE)

Number of GPUs available: 1
Found cached images, checking validity...
Cached images valid — proceeding.


In [4]:
##############################################################################################
##############################################################################################
# Paths
train_set_path = CACHED_IMAGES_PATH / 'train'
val_set_path = CACHED_IMAGES_PATH / 'val'
test_set_path = CACHED_IMAGES_PATH / 'test'

# Datasets
dataset_train = CachedDataset(  train_set_path,
                                label_criterion = (None, 'soft'))
dataset_val =   CachedDataset(  val_set_path,
                                label_criterion = (None, 'soft'))
dataset_test =  CachedDataset(  test_set_path,
                                label_criterion = (None, 'soft'))
# Dataloaders
train_dataloader =  DataLoader( dataset_train,
                                batch_size = BATCH_SIZE,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = True)
val_dataloader =    DataLoader( dataset_val,
                                batch_size = BATCH_SIZE,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = False)
test_dataloader =   DataLoader( dataset_test,
                                batch_size = BATCH_SIZE,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = False)

In [5]:
##############################################################################################
##############################################################################################
model = build_model(BACKBONE_DICT,
                    CLASSIFIER_DICT)
model.to(device)

# Separate parameter groups for adjusted learning rate
temporal_proc_params, head_params = get_parameter_groups(model, CLASSIFIER_DICT)

optimizer = optim.AdamW(
    [
        {"params": temporal_proc_params, "lr": TEMPORAL_PROC_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

# Calculate prior alpha
if BB_LOSS_VARIABLES['use_kl'] and BB_LOSS_VARIABLES['prior_alpha'] is not None:
    BB_LOSS_VARIABLES['prior_alpha'] = {'C1': ((1-BB_LOSS_VARIABLES['prior_alpha']['pi_C1'])*BB_LOSS_VARIABLES['prior_alpha']['nu'], BB_LOSS_VARIABLES['prior_alpha']['pi_C1']*BB_LOSS_VARIABLES['prior_alpha']['nu']),
                                        'C2': ((1-BB_LOSS_VARIABLES['prior_alpha']['pi_C2'])*BB_LOSS_VARIABLES['prior_alpha']['nu'], BB_LOSS_VARIABLES['prior_alpha']['pi_C2']*BB_LOSS_VARIABLES['prior_alpha']['nu']),
                                        'C3': ((1-BB_LOSS_VARIABLES['prior_alpha']['pi_C3'])*BB_LOSS_VARIABLES['prior_alpha']['nu'], BB_LOSS_VARIABLES['prior_alpha']['pi_C3']*BB_LOSS_VARIABLES['prior_alpha']['nu'])}


[SwinV2] Stage freeze summary:
  stage 0  ( 2 blocks, dim= 128)  →  FROZEN
  stage 1  ( 2 blocks, dim= 256)  →  FROZEN
  stage 2  (18 blocks, dim= 512)  →  trainable
  stage 3  ( 2 blocks, dim=1024)  →  trainable
[SwinV2] Params — total: 86,896,891  |  trainable: 84,765,027  (97.55 %)


In [ ]:
##############################################################################################
##############################################################################################
results_dict = {}

best_bacc_across_epochs = -1.0
best_epoch = 0

for epoch in range(EPOCHS):
        print(f"Epoch: {epoch+1:02}/{EPOCHS:02}")
        print("Training")
        train_loss_sum = 0.0
        len_train_loader = len(train_dataloader)
        train_output_dict = {   'C1': { 'probs':     [],
                                        'preds':     [],
                                        'uncerts':   []},
                                'C2': { 'probs':     [],
                                        'preds':     [],
                                        'uncerts':   []},
                                'C3': { 'probs':     [],
                                        'preds':     [],
                                        'uncerts':   []},
                                'labels':            [],
                                'vid_ids':           [],
                                'frame_ids':         []}
        
        model.train()
        for idx, (images, labels, vid_id, frame_id) in enumerate(train_dataloader):
                print(f'\r{idx+1}/{len_train_loader}', end='', flush=True)

                images, labels = images.to(device), labels.to(device)
                torch.cuda.synchronize()

                optimizer.zero_grad()

                output = model(images)

                train_loss_total = total_bb_loss(output,
                                                 labels,
                                                 weights = BB_LOSS_VARIABLES['weights_bb_loss'],
                                                 use_kl = BB_LOSS_VARIABLES['use_kl'],
                                                 prior_alpha = BB_LOSS_VARIABLES['prior_alpha'])
                
                train_loss_total.backward()
                optimizer.step()

                # Populate the output dict with probs, preds, and uncerts per sample
                train_output_dict = update_model_output_dict(output, train_output_dict, evidential = True)
                train_output_dict['labels'].append(labels.detach().cpu())
                train_output_dict['vid_ids'].append(vid_id)
                train_output_dict['frame_ids'].append(frame_id)
                train_loss_sum += train_loss_total.item()

        results, train_output_dict = calculate_metrics(train_output_dict, evidential = True)

        avg_train_loss = train_loss_sum / len_train_loader
        results['loss'] = round(avg_train_loss, 4)
        
        print(f"\n--- Training Metrics ---")
        print(f"Train Avg Accuracy              {results['avg_accuracy']:.4f}")
        print(f"Train Avg BAcc                  {results['avg_bacc']:.4f}")
        print(f"Train mAP                       {results['mAP']:.4f}")
        print(f"Train Uncert. Pos:              {results['avg_uncert_pos']:.4f}")
        print(f"Train Uncert. Neg:              {results['avg_uncert_neg']:.4f}")
        print(f"Train Loss:                     {results['loss']:.4f}\n")
        print(f"Train C1 Accuracy               {results['accuracy_C1']:.4f}")
        print(f"Train C2 Accuracy               {results['accuracy_C2']:.4f}")
        print(f"Train C3 Accuracy               {results['accuracy_C3']:.4f}\n")
        print(f"Train C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
        print(f"Train C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
        print(f"Train C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
        print(f"Train C1 AP:                    {results['ap_C1']:.4f}")
        print(f"Train C2 AP:                    {results['ap_C2']:.4f}")
        print(f"Train C3 AP:                    {results['ap_C3']:.4f}")
        print(f"------------------------\n")
        results_dict[f"Epoch {epoch+1} Train"] = results
        
        print('Validation')
        val_loss_sum = 0.0
        len_val_loader = len(val_dataloader)
        val_output_dict = {     'C1': { 'probs':     [],
                                        'preds':     [],
                                        'uncerts':   []},
                                'C2': { 'probs':     [],
                                        'preds':     [],
                                        'uncerts':   []},
                                'C3': { 'probs':     [],
                                        'preds':     [],
                                        'uncerts':   []},
                                'labels':            [],
                                'vid_ids':           [],
                                'frame_ids':         []}
        model.eval() 
        with torch.inference_mode():
                for idx, (images, labels, vid_id, frame_id) in enumerate(val_dataloader):
                        print(f'\r{idx+1}/{len_val_loader}', end='', flush=True)
                        images, labels = images.to(device), labels.to(device)
                        torch.cuda.synchronize()

                        output = model(images)

                        val_loss_total = total_bb_loss( output,
                                                        labels,
                                                        weights = BB_LOSS_VARIABLES['weights_bb_loss'],
                                                        use_kl = BB_LOSS_VARIABLES['use_kl'],
                                                        prior_alpha = BB_LOSS_VARIABLES['prior_alpha'])
                        
                        val_output_dict = update_model_output_dict(output, val_output_dict, evidential = True)
                        val_output_dict['labels'].append(labels.detach().cpu())
                        val_output_dict['vid_ids'].append(vid_id)
                        val_output_dict['frame_ids'].append(frame_id)
                        val_loss_sum += val_loss_total.item()

        results, val_output_dict = calculate_metrics(val_output_dict, evidential = True)
        avg_val_loss = val_loss_sum / len_val_loader
        results['loss'] = round(avg_val_loss, 4)
        print(f"\n--- Validation Metrics ---")
        print(f"Val Avg Accuracy              {results['avg_accuracy']:.4f}")
        print(f"Val Avg BAcc                  {results['avg_bacc']:.4f}")
        print(f"Val mAP                       {results['mAP']:.4f}")
        print(f"Val Uncert. Pos:              {results['avg_uncert_pos']:.4f}")
        print(f"Val Uncert. Neg:              {results['avg_uncert_neg']:.4f}")
        print(f"Val Loss:                     {results['loss']:.4f}\n")
        print(f"Val C1 Accuracy               {results['accuracy_C1']:.4f}")
        print(f"Val C2 Accuracy               {results['accuracy_C2']:.4f}")
        print(f"Val C3 Accuracy               {results['accuracy_C3']:.4f}\n")
        print(f"Val C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
        print(f"Val C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
        print(f"Val C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
        print(f"Val C1 AP:                    {results['ap_C1']:.4f}")
        print(f"Val C2 AP:                    {results['ap_C2']:.4f}")
        print(f"Val C3 AP:                    {results['ap_C3']:.4f}")
        print(f"------------------------\n")

        results['saved'] =      {'C1': {'probs':     val_output_dict['C1']['probs'].tolist(),
                                        'preds':     val_output_dict['C1']['preds'].tolist(),
                                        'uncerts':   val_output_dict['C1']['uncerts'].tolist()},
                                 'C2': {'probs':     val_output_dict['C2']['probs'].tolist(),
                                        'preds':     val_output_dict['C2']['preds'].tolist(),
                                        'uncerts':   val_output_dict['C2']['uncerts'].tolist()},
                                 'C3': {'probs':     val_output_dict['C3']['probs'].tolist(),
                                        'preds':     val_output_dict['C3']['preds'].tolist(),
                                        'uncerts':   val_output_dict['C3']['uncerts'].tolist()},
                                 'labels':           val_output_dict['labels'].tolist(),
                                 'vid_ids':          val_output_dict['vid_ids'].tolist(),
                                 'frame_ids':        val_output_dict['frame_ids'].tolist()}
        results_dict[f"Epoch {epoch+1} Val"] = results

        # Save results
        with open(PWD / 'results' / f'{EXPERIMENT_NAME}_results.json', 'w') as file:
                json.dump(results_dict, file, indent=4)

        # Save weights of the best epoch
        if results['avg_bacc'] >= best_bacc_across_epochs:
                best_bacc_across_epochs = results['avg_bacc']
                best_epoch = epoch+1
                print(f"New best result (Epoch {best_epoch}), saving weights...")
                weights_path = Path.cwd() / 'weights'
                checkpoint_dir = os.path.join(weights_path, f'{EXPERIMENT_NAME}.pt')
                torch.save(model.state_dict(), checkpoint_dir)
        else:
                print('\n')

print(f"Testing @ epoch {best_epoch}")
test_loss_sum = 0.0
len_test_loader = len(test_dataloader)
test_output_dict = {    'C1': { 'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                        'C2': { 'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                        'C3': { 'probs':     [],
                                'preds':     [],
                                'uncerts':   []},
                        'labels':            [],
                        'vid_ids':           [],
                        'frame_ids':         []} 
checkpoint = torch.load(checkpoint_dir, map_location=device)
model.load_state_dict(checkpoint)
model.to(device)
model.eval()
with torch.inference_mode():
    for idx, (images, labels, vid_id, frame_id) in enumerate(test_dataloader):
            print(f'\r{idx+1}/{len_test_loader}', end='', flush=True)
            images, labels = images.to(device), labels.to(device)
            torch.cuda.synchronize()

            output = model(images)

            test_loss_total = total_bb_loss(    output,
                                                labels,
                                                weights = BB_LOSS_VARIABLES['weights_bb_loss'],
                                                use_kl = BB_LOSS_VARIABLES['use_kl'],
                                                prior_alpha = BB_LOSS_VARIABLES['prior_alpha'])
        
            test_output_dict = update_model_output_dict(output, test_output_dict, evidential = True)
            test_output_dict['labels'].append(labels.detach().cpu())
            test_output_dict['vid_ids'].append(vid_id)
            test_output_dict['frame_ids'].append(frame_id)
            test_loss_sum += test_loss_total.item()

results, test_output_dict = calculate_metrics(test_output_dict, evidential = True)
avg_test_loss = test_loss_sum / len_test_loader
results['loss'] = round(avg_test_loss, 4)

print(f"\n--- Testing Metrics ---")
print(f"Test Avg Accuracy              {results['avg_accuracy']:.4f}")
print(f"Test Avg BAcc                  {results['avg_bacc']:.4f}")
print(f"Test mAP                       {results['mAP']:.4f}")
print(f"Test Uncert. Pos:              {results['avg_uncert_pos']:.4f}")
print(f"Test Uncert. Neg:              {results['avg_uncert_neg']:.4f}")
print(f"Test Loss:                     {results['loss']:.4f}\n")
print(f"Test C1 Accuracy               {results['accuracy_C1']:.4f}")
print(f"Test C2 Accuracy               {results['accuracy_C2']:.4f}")
print(f"Test C3 Accuracy               {results['accuracy_C3']:.4f}\n")
print(f"Test C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
print(f"Test C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
print(f"Test C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
print(f"Test C1 AP:                    {results['ap_C1']:.4f}")
print(f"Test C2 AP:                    {results['ap_C2']:.4f}")
print(f"Test C3 AP:                    {results['ap_C3']:.4f}")
print(f"------------------------\n")
results['saved'] = {'C1': {'probs':      test_output_dict['C1']['probs'].tolist(),
                            'preds':     test_output_dict['C1']['preds'].tolist(),
                            'uncerts':   test_output_dict['C1']['uncerts'].tolist()},
                    'C2': {'probs':      test_output_dict['C2']['probs'].tolist(),
                            'preds':     test_output_dict['C2']['preds'].tolist(),
                            'uncerts':   test_output_dict['C2']['uncerts'].tolist()},
                    'C3': {'probs':      test_output_dict['C3']['probs'].tolist(),
                            'preds':     test_output_dict['C3']['preds'].tolist(),
                            'uncerts':   test_output_dict['C3']['uncerts'].tolist()},
                    'labels':            test_output_dict['labels'].tolist(),
                    'vid_ids':           test_output_dict['vid_ids'].tolist(),
                    'frame_ids':         test_output_dict['frame_ids'].tolist()}

results_dict[f"Epoch {best_epoch} Test"] = results
with open(PWD / 'results' / f'{EXPERIMENT_NAME}_results.json', 'w') as file:
    json.dump(results_dict, file, indent=4)

Epoch: 01/10
Training
7/870
--- Training Metrics ---
Train Avg Accuracy              0.8155
Train Avg BAcc                  0.6635
Train mAP                       0.3713
Train Uncert. Pos:              0.5904
Train Uncert. Neg:              0.5742
Train Loss:                     0.0290

Train C1 Accuracy               0.7500
Train C2 Accuracy               0.9107
Train C3 Accuracy               0.7857

Train C1 Balanced Accuracy:     0.5510
Train C2 Balanced Accuracy:     0.8365
Train C3 Balanced Accuracy:     0.6028

Train C1 AP:                    0.2875
Train C2 AP:                    0.4613
Train C3 AP:                    0.3652
------------------------

Validation
7/292
--- Validation Metrics ---
Val Avg Accuracy              1.0000
Val Avg BAcc                  0.5000
Val mAP                       0.0000
Val Uncert. Pos:              nan
Val Uncert. Neg:              0.5190
Val Loss:                     0.0401

Val C1 Accuracy               1.0000
Val C2 Accuracy               1.

KeyboardInterrupt: 

In [ ]:
images.shape

torch.Size([8, 5, 3, 384, 384])